<a href="https://colab.research.google.com/github/NataMaru/ML_for_people_tasks/blob/main/HW_%D0%9C%D0%B5%D1%82%D0%BE%D0%B4%D0%B8_%D0%BF%D0%BE%D0%BD%D0%B8%D0%B6%D0%B5%D0%BD%D0%BD%D1%8F_%D1%80%D0%BE%D0%B7%D0%BC%D1%96%D1%80%D0%BD%D0%BE%D1%81%D1%82%D1%96.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Домашнє завдання: Пониження розмірностей для Аналізу Портретів Клієнтів

#### Контекст
В цьому ДЗ ми попрацюємо з методами пониження розмірності на наборі даних для задачі аналізу портретів клієнтів (Customer Personality Analysis). **В попередньому ДЗ ми працювали з цими даними використовуючи кластеризацію, зараз використаємо кластеризацію і візуалізауємо результати з різними методами.**

Customer Personality Analysis - це аналіз різних сегментів клієнтів компанії. Цей аналіз дозволяє бізнесу краще розуміти своїх клієнтів і полегшує процес адаптації продуктів під конкретні потреби, поведінку та інтереси різних типів клієнтів.

Аналіз портретів клієнтів допомагає бізнесу змінювати свій продукт на основі цільової аудиторії, розділеної на різні сегменти. Наприклад, замість того, щоб витрачати гроші на маркетинг нового продукту для всіх клієнтів у базі даних компанії, бізнес може проаналізувати, який сегмент клієнтів найімовірніше придбає продукт, і потім зосередити маркетингові зусилля лише на цьому сегменті.

#### Вхідні дані
Вам надано набір даних з такими атрибутами:

**Характеристики користувачів:**
- `ID`: Унікальний ідентифікатор клієнта
- `Year_Birth`: Рік народження клієнта
- `Education`: Рівень освіти клієнта
- `Marital_Status`: Сімейний стан клієнта
- `Income`: Річний дохід домогосподарства клієнта
- `Kidhome`: Кількість дітей у домогосподарстві клієнта
- `Teenhome`: Кількість підлітків у домогосподарстві клієнта
- `Dt_Customer`: Дата реєстрації клієнта у компанії
- `Recency`: Кількість днів з моменту останньої покупки клієнта
- `Complain`: 1, якщо клієнт скаржився за останні 2 роки, 0 - якщо ні

**Продукти:**
- `MntWines`: Сума, витрачена на вино за останні 2 роки
- `MntFruits`: Сума, витрачена на фрукти за останні 2 роки
- `MntMeatProducts`: Сума, витрачена на м'ясні продукти за останні 2 роки
- `MntFishProducts`: Сума, витрачена на рибні продукти за останні 2 роки
- `MntSweetProducts`: Сума, витрачена на солодощі за останні 2 роки
- `MntGoldProds`: Сума, витрачена на золото за останні 2 роки

**Акції:**
- `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
- `AcceptedCmp1`: 1, якщо клієнт прийняв пропозицію у першій кампанії, 0 - якщо ні
- `AcceptedCmp2`: 1, якщо клієнт прийняв пропозицію у другій кампанії, 0 - якщо ні
- `AcceptedCmp3`: 1, якщо клієнт прийняв пропозицію у третій кампанії, 0 - якщо ні
- `AcceptedCmp4`: 1, якщо клієнт прийняв пропозицію у четвертій кампанії, 0 - якщо ні
- `AcceptedCmp5`: 1, якщо клієнт прийняв пропозицію у п'ятій кампанії, 0 - якщо ні
- `Response`: 1, якщо клієнт прийняв пропозицію в останній кампанії, 0 - якщо ні

**Взаємодія з компанією:**
- `NumWebPurchases`: Кількість покупок, зроблених через вебсайт компанії
- `NumCatalogPurchases`: Кількість покупок, зроблених за каталогом
- `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
- `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць


Для початку, запустіть код нижче. Всі ці кроки ми робили в попередньому ДЗ і для того, щоб результати кластеризації у нас були схожими, потрібно аби передобробка була однаковою.

In [1]:
import pandas as pd
from sklearn.manifold import TSNE
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import plotly.express as px
from sklearn.preprocessing import StandardScaler


In [2]:
# 1. Завантаження даних
df = pd.read_csv('marketing_campaign.csv', sep='\t')

# 2. Обробка пропущених значень
df['Income_not_filled'] = df.Income.isna()
df.Income = df.Income.fillna(-1)

# 3. Обробка дати реєстрації
df.Dt_Customer = pd.to_datetime(df.Dt_Customer, format='%d-%m-%Y')
today = df.Dt_Customer.max()
df['days_lifetime'] = (today - df.Dt_Customer).dt.days
df['years_customer'] = df.Year_Birth.apply(lambda x: today.year - x)

# 4. Категоризація рівня освіти
df_education = pd.get_dummies(df.Education, prefix='education').astype(int)
df = pd.concat([df, df_education], axis=1)

# 5. Очищення сімейного стану
marital_status_map = {'Alone': 'Single', 'Absurd': 'Else', 'YOLO': 'Else'}
df['Marital_Status_clean'] = df.Marital_Status.map(marital_status_map)
df_ms = pd.get_dummies(df.Marital_Status_clean, prefix='marital').astype(int)
df = pd.concat([df, df_ms], axis=1)

# 6. Форматування доходу і видалення викиду
df.Income = df.Income.astype(int)
df = df[df.Income != 666666]

# 7. Створення фінального набору даних
X = df.drop(['ID', 'Dt_Customer', 'Education', 'Marital_Status', 'Marital_Status_clean'], axis=1)
X.reset_index(drop=True, inplace=True)

### Завдання 1: Виконання кластеризації та пониження розмірності для візуалізації результатів

Ваше завдання — провести кластеризацію клієнтів та візуалізувати результати кластеризації, використовуючи метод головних компонент (PCA) для пониження розмірності даних.

#### Інструкції:

1. **Вибір ключових характеристик:**
   Давайте обмежимось тільки наступними хараткеристиками для кластеризації цього разу:
   - `Income`: Річний дохід домогосподарства клієнта
   - `Recency`: Кількість днів з моменту останньої покупки клієнта
   - `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
   - `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
   - `days_lifetime`: Кількість днів з моменту реєстрації клієнта у компанії
   - `years_customer`: Вік клієнта
   - `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць
   Відберіть в наборі даних `X` лише ці характеристики.

2. **Стандартизація даних:**
   Використайте метод `StandardScaler` для стандартизації значень обраних характеристик.
   
   **Чому не MinMaxScaler:**
   - Для PCA краще використовувати StandardScaler, бо він вирівнює дисперсію ознак, на відміну від MinMaxScaler, що просто масштабує значення без врахування варіації.

   - Для K-Means також краще використовувати StandardScaler, бо алгоритм чутливий до масштабів: фічі з більшими значеннями сильніше впливають на обчислення відстаней, що може спотворити кластери.

3. **Кластеризація:**
   Проведіть кластеризацію клієнтів, використовуючи метод `KMeans` з трьома кластерами.

4. **Пониження розмірності:**
   Використайте метод головних компонент (PCA) для пониження розмірності даних до трьох компонент.

5. **Візуалізація результатів:**
   Використовуючи plolty express побудуйте 3D-графік розподілу клієнтів у просторі трьох головних компонент, де кольором позначено кластери.

6. **Опишіть, що спостерігаєте:**
   Чи кластеризація чітко розділила дані?

Далі ми детальніше проінтерпретуємо результати візуалізації і пониження розмірностей.

In [3]:
features = [
    'Income', 'Recency', 'NumStorePurchases', 'NumDealsPurchases',
    'days_lifetime', 'years_customer', 'NumWebVisitsMonth'
]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


X_scaled_df = pd.DataFrame(X_scaled, columns=features)

In [4]:
X_scaled_df.head()

,Income,Recency,NumStorePurchases,NumDealsPurchases,days_lifetime,years_customer,NumWebVisitsMonth
0,0.304796,0.306624,-0.551136,0.349782,1.530940,0.984922,0.693887
1,-0.229724,-0.384051,-1.166440,-0.167840,-1.190204,1.235281,-0.130311
2,0.915500,-0.798456,1.294778,-0.685461,-0.205644,0.317300,-0.542410
3,-1.122463,-0.798456,-0.551136,-0.167840,-1.061568,-1.268304,0.281788
4,0.311820,1.549838,0.064169,1.385025,-0.952722,-1.017946,-0.130311


In [5]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=3)
pca_result = pca.fit_transform(X_scaled)


pca_df = pd.DataFrame(
    data=pca_result,
    columns=['PC1', 'PC2', 'PC3']
)
pca_df['Cluster'] = clusters.astype(str)

fig = px.scatter_3d(
    pca_df,
    x='PC1', y='PC2', z='PC3',
    color='Cluster',
    title='3D Кластеризація клієнтів (PCA)',
    labels={'PC1': 'Головна компонента 1', 'PC2': 'Головна компонента 2', 'PC3': 'Головна компонента 3'},
    opacity=0.7
)

fig.show()

на візуалізації видно, що кластери перетинаються між собою, але загалом непогано розподілені.

### Завдання 2: Аналіз результатів пониження розмірності

1. **Розрахунок частки поясненої дисперсії:**
   Визначте, яка частка загальної варіації даних пояснюється кожною з трьох головних компонент (PC1, PC2, PC3) за допомогою атрибуту `explained_variance_ratio_` об'єкта PCA. Виведіть результат на екран.

2. **Розрахунок кумулятивної частки поясненої дисперсії:**
   Обчисліть кумулятивну частку поясненої дисперсії для трьох головних компонент, щоб зрозуміти, скільки варіації даних пояснюється першими кількома компонентами.

In [6]:
explained_variance = pca.explained_variance_ratio_

print("Частка поясненої дисперсії для кожної компоненти:")
for i, variance in enumerate(explained_variance):
    print(f"PC{i+1}: {variance:.2%}")

cumulative_variance = explained_variance.sum()

print(f"\nЗагальна (кумулятивна) дисперсія для 3-х компонент: {cumulative_variance:.2%}")


Частка поясненої дисперсії для кожної компоненти:
PC1: 31.85%
PC2: 19.66%
PC3: 14.34%

Загальна (кумулятивна) дисперсія для 3-х компонент: 65.85%


### Завдання 3: Інтерпретація "Loadings"

Продовжуємо інтерпретацію результатів `PCA`і познайомимось з новим поняттям `loadings`, яке допоможе нам знайти звʼязок між головними компонентами і оригінальними ознаками в наборі даних.

Ми зараз побудували візуалізацію кластерів точок даних в просторі трьох головних компонент. Але хочеться знайти звʼязок між головними компонентами і оригінальними ознаками. Для розуміння, які початкові характеристики даних мають найбільший вплив на ці головні компоненти, ми можемо використати атрибут `components_` методу `PCA`.

#### Що таке `pca.components_`?

`pca.components_` — це масив, який містить коефіцієнти (або "ваги"), що показують внесок кожної вихідної ознаки у кожну з головних компонент. Ці коефіцієнти ще називаються **"loading"** або "навантаженнями" компонент.

- **Loadings** (`навантаження`) відображають важливість кожної змінної (ознаки) для відповідної головної компоненти. Вони показують, яким чином змінні поєднуються, щоб утворити нові, зменшені вимірювання.
- Якщо коефіцієнт має високе абсолютне значення (як позитивне, так і негативне), це вказує на те, що відповідна змінна сильно впливає на головну компоненту.

#### Саме завдання
Ваше завдання — обчислити "навантаження" для кожної з головних компонент і інтерпретувати результати.

1. **Обчислення loadings для компонент:**
   Використайте атрибут `components_` об'єкта PCA для створення DataFrame, який відображатиме внесок кожної вихідної ознаки в кожну головну компоненту.

2. **Інтерпретація результатів:**
   Виведіть значення "навантажень" і проаналізуйте, які ознаки найбільше впливають на кожну головну компоненту.

In [7]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2', 'PC3'],
    index=features
)

print("Навантаження (loadings) для головних компонент:")
display(loadings.style.background_gradient(cmap='coolwarm'))


Навантаження (loadings) для головних компонент:


,PC1,PC2,PC3
Income,0.584238,0.165255,-0.044491
Recency,0.010075,0.036810,0.975799
NumStorePurchases,0.488621,0.384659,-0.080566
NumDealsPurchases,-0.198661,0.608119,-0.034808
days_lifetime,-0.132683,0.602585,-0.022223
years_customer,0.189393,0.208946,0.193368
NumWebVisitsMonth,-0.571728,0.216329,-0.015815


висновок
1. PC1:  ця компонента розподіляє клієнтів за рівнем доходу та поведінкою онлайн/оффлайн. Сильний позитивний вплив Income та NumStorePurchases, та сильний негативний NumWebVisitsMonth -  це вказує на те, що при високому доході та частому відвудуванні офф магазинів, відвідування сайту знижується.
2. PC2: сильний позитивний вплив NumDealsPurchases та days_lifetime - вказує на те що це вісь ояльних гостей, що полюбляють знижки та вміло їх використовують
3. PC3: компонента показує давність покупок, тобто дає змогу виділити активних, сплячих та черн сегмент гостей, оскільки сильний вплив, що майже дорівнює 1, має ознака Recency.


###Завдання 4
Давайте проаналізуємо "навантаження" (**loadings**) для трьох головних компонент після вилучення ознаки `Income`. Це допоможе нам зрозуміти, як змінилася важливість інших ознак для кожної головної компоненти, коли одна з ключових характеристик (`Income`) була вилучена.

#### Кроки для проведення аналізу і ваше завдання:

1. Видаліть ознаку `Income` з нашого набору даних `X` і повторно виконайте PCA (метод головних компонент) для отримання нових "навантажень".

2. Обчисліть нові "навантаження" для трьох головних компонент на наборі даних без `Income`

3. Проаналізуйте, які ознаки мають найбільший вплив на кожну головну компоненту після вилучення `Income`.

4. Перегляньте, наскільки кожна з головних компонент пояснює дисперсію в даних без ознаки `Income`.

In [8]:
new_features = [ 'Recency', 'NumStorePurchases', 'NumDealsPurchases',
    'days_lifetime', 'years_customer', 'NumWebVisitsMonth'
]
X_no_inc = df[new_features]

scaler = StandardScaler()
X_scaled_no_inc = scaler.fit_transform(X_no_inc)


X_scaled_no_inc_df = pd.DataFrame(X_scaled_no_inc, columns=new_features)

pca_new = PCA(n_components=3)
pca_result_new = pca_new.fit_transform(X_scaled_no_inc_df)


loadings_new = pd.DataFrame(
    pca_new.components_.T,
    columns=['PC1', 'PC2', 'PC3'],
    index=new_features
)

print(f"Нова кумулятивна дисперсія: {pca_new.explained_variance_ratio_.sum():.2%}")
display(loadings_new.style.background_gradient(cmap='coolwarm'))

Нова кумулятивна дисперсія: 65.52%


,PC1,PC2,PC3
Recency,-0.014416,0.074880,0.990219
NumStorePurchases,-0.381237,0.629202,-0.111063
NumDealsPurchases,0.445591,0.461539,-0.061793
days_lifetime,0.383782,0.467914,0.006776
years_customer,-0.177270,0.393576,0.056285
NumWebVisitsMonth,0.690787,-0.107863,0.009910


після видалення поля income розподіл навантаження змінився оскільки це поле було найсильнішим фактором
1. PC1 - тепер це компонента онлайн покупок та акцій
2. PC2 - тепер це компонента оффлайн покупок та досвіду
3. PC3 -  майже не змінилась, давнина покупок все ще має найбільший вплив - виглядає так наче ознака Recency  завжди буде мати "свою власну" компоненту

In [9]:
explained_variance_new = pca_new.explained_variance_ratio_

print("Частка поясненої дисперсії для кожної компоненти:")
for i, variance in enumerate(explained_variance_new):
    print(f"PC{i+1}: {variance:.2%}")

cumulative_variance_new = explained_variance_new.sum()

print(f"\nЗагальна (кумулятивна) дисперсія для 3-х компонент: {cumulative_variance_new:.2%}")

Частка поясненої дисперсії для кожної компоненти:
PC1: 27.39%
PC2: 21.44%
PC3: 16.68%

Загальна (кумулятивна) дисперсія для 3-х компонент: 65.52%


In [10]:
index = ['PC1', 'PC2', 'PC3', 'Total (Cumulative)']

data = {
    'Original PCA': list(explained_variance) + [sum(explained_variance)],
    'New PCA': list(explained_variance_new) + [sum(explained_variance_new)]
}

comparison_df = pd.DataFrame(data, index=index)

print("Порівняння поясненої дисперсії:")
display(comparison_df.style.format("{:.2%}"))

Порівняння поясненої дисперсії:


,Original PCA,New PCA
PC1,31.85%,27.39%
PC2,19.66%,21.44%
PC3,14.34%,16.68%
Total (Cumulative),65.85%,65.52%


Після видалення income комулятивна дисперсія майже не впала, що свідчить про те що ознака частково дублювалась іншими.


### Завдання 5: Візуалізація кластеризації за допомогою t-SNE

Ваше завдання — використати метод t-SNE для візуалізації результатів кластеризації клієнтів у двовимірному просторі. Метод t-SNE допомагає знизити розмірність даних та зберегти локальні структури в даних, що робить його ефективним для візуалізації високорозмірних даних. Ми також зможемо порівняти результат цього методу з РСА.

1. Використайте метод t-SNE для зниження розмірності до 2х вимірів даних, які включають ознаки всі, що і в завданні 1, а також були відмасштабовані перед пониженням розмірностей.

2. Створіть новий DataFrame з координатами, отриманими після застосування t-SNE, та додайте до нього мітки кластерів.

3. Побудуйте інтерактивний 2D-графік розподілу клієнтів, де кольором буде позначено різні кластери і проаналізуйте графік з рекомендаціями нижче (можливо треба буде вивести додаткові візуалізації чи таблиці для інтерпретації, але треба прям зрозуміти, які ознаки формують який кластер і чим кластери відрізняються одне від одного).

  **Опишіть отримані кластери з точки зору ознак.**

4. Опишіть відмінність графіка tSNE від PCA.

#### ЯК можна інтерпретувати з t-SNE?

Хоча t-SNE не надає "компонентів" як РСА, він забезпечує низьковимірне представлення даних, яке можна візуально інтерпретувати:

- **Кластери:** t-SNE особливо добре показує кластери подібних точок. Якщо ви бачите чітко визначені кластери на графіку t-SNE, це свідчить про наявність груп схожих спостережень у ваших даних. Проаналізувати їх можемо, якщо додамо дані в `hover_data` або якщо якісь з даних виведемо як розмір чи форма точок на візуалізації. Також корисно може бути вивести середні значення ознак по кластерам.
- **Локальна структура:** Відносне розташування точок одного кластеру на графіку t-SNE може допомогти вам зрозуміти, які дані подібні між собою.
- **Глобальна структура:** Будьте обережні; t-SNE менш надійний для відображення глобальних структур (наприклад, відстаней між кластерами) у порівнянні з PCA, бо t-SNE націлений на збереження саме локальних структур.

In [11]:
tsne = TSNE(n_components=2, random_state=42, perplexity=50)
tsne_results = tsne.fit_transform(X_scaled)

df_tsne = pd.DataFrame(tsne_results, columns=['tsne_1', 'tsne_2'])


kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_tsne['cluster'] = kmeans.fit_predict(X_scaled)

df_tsne = pd.concat([df_tsne, X.reset_index(drop=True)], axis=1)

fig = px.scatter(
    df_tsne, x='tsne_1', y='tsne_2',
    color='cluster',
    title='t-SNE візуалізація кластерів клієнтів',
    hover_data=features,
    color_continuous_scale='Viridis'
)
fig.show()

###висновки
1. на графіку видно чітко три кластери, тобто дані дійсно можна розподілити на три кластери
2. 1 і 2 кластери ( жовтий та бірюзовий) більш щільні ніж 0 кластер( фіолетовий)

In [12]:
#подивимось на портрет профілей в кластерах
cluster_profile = df_tsne.groupby('cluster')[features].median()

cluster_profile['Customer_Count'] = df_tsne.groupby('cluster').size()
cluster_profile = cluster_profile.round(2)

import IPython.display as display
display.display(cluster_profile)

,Income,Recency,NumStorePurchases,NumDealsPurchases,days_lifetime,years_customer,NumWebVisitsMonth,Customer_Count
cluster,,,,,,,,
0,33456.0,48.0,3.0,2.0,298.0,41.0,7.0,961
1,72025.0,50.0,8.0,1.0,293.0,47.0,3.0,823
2,52852.0,51.0,6.0,5.0,543.0,48.0,7.0,455


згідно таблиці профілей вище можна сказати, що:
1. кластер 0 - це гості з невеликим доходом, що рідко купують,але часто продивляються інформацію на сайті, також вони є відносними новачками
2. кластер 1- оффлайнові VIP гості, що мають високий дохід, часто купують оффлайн та рідко відвідаюсь сайт
3. кластер 2 - це лояльні гості, що купують давно та стабільно і полюбляють знижечки.

Варто також зауважити: якщо подивитись на графік то кластер 0( фіолетовий)не такий щільний як інші і тому його можна було б ще якось розділити.